In [1]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Steam Game Recommender — Item-Item Collaborative Filtering (v3)

**Input:** `output/steam_merged.csv`  
**Outputs:** `output/recommendations.json`, `output/games_metadata.json`

## Why item-item instead of user-user?

| | User-User CF (v1/v2) | Item-Item CF (v3) |
|---|---|---|
| Similarity matrix size | 3,270 × 3,270 users | 791 × 791 popular games |
| Co-rating density | Very sparse — most user pairs share few games | Much denser — popular items are rated by many users |
| Stability | Unstable; one user with unusual taste skews a cluster | Stable; item similarity averages over many users |
| RMSE result (v2) | Worse than baseline | Should beat baseline |

**Algorithm:**
```
1. Build item-item Pearson similarity matrix (791 × 791)
2. For each user u and candidate game i (unplayed, popular):
   - Find games j that u HAS played and that are similar to i
   - pred(u,i) = Σ[ sim(i,j) * (r(u,j) - b(u,j)) ] / Σ|sim(i,j)|  +  b(u,i)
3. Rank candidates by predicted score → top 10
```
Item-item CF predicts how much you'll like a new game based on how similar it is
to games you've already played and rated — much more stable than finding similar users.

## 1. Imports & Config

In [2]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from collections import Counter
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

INPUT_PATH = Path('output/steam_merged.csv')
REC_PATH   = Path('output/recommendations.json')
META_PATH  = Path('output/games_metadata.json')

MIN_RATING        = 3    # ratings >= this used for similarity matrix
MIN_GAMES_COMMON  = 3    # min users who rated both items before trusting item similarity
TOP_K_ITEMS       = 20   # top-K similar items to use per prediction
TOP_N             = 10   # recommendations per user
MIN_GAME_RATERS   = 5    # game must have been rated by >= this many users
TEST_SIZE         = 0.20

print('Config ready.')

Config ready.


## 2. Load Data

In [3]:
df = pd.read_csv(INPUT_PATH)
print(f'Loaded {len(df):,} rows  |  {df["user_id"].nunique():,} users  |  {df["game"].nunique():,} games')
all_played_by_user = df.groupby('user_id')['game'].apply(set).to_dict()
df.head(3)

Loaded 61,280 rows  |  3,466 users  |  3,564 games


,user_id,game,hours_played,rating,app_id,game_title,release_date,price,description,cover_image_url,positive_reviews,negative_reviews,avg_playtime_mins,developers,publishers,categories,genres,tags,review_score_pct,total_reviews
0,151603712,The Elder Scrolls V Skyrim,273.0,5,72850.0,The Elder Scrolls V: Skyrim,"Nov 10, 2011",19.99,EPIC FANTASY REBORN The next chapter in the hi...,https://shared.akamai.steamstatic.com/store_it...,300915.0,16364.0,7178.0,Bethesda Game Studios,Bethesda Softworks,"Single-player,Steam Achievements,Steam Trading...",RPG,"Open World,RPG,Fantasy,Adventure,Dragons,Modda...",94.8,317279.0
1,151603712,Fallout 4,87.0,4,377160.0,Fallout 4,"Nov 9, 2015",7.99,"From Bethesda Game Studios, the award-winning ...",https://shared.akamai.steamstatic.com/store_it...,326054.0,66233.0,6128.0,Bethesda Game Studios,Bethesda Softworks,"Single-player,Steam Achievements,Full controll...",RPG,"Open World,Post-apocalyptic,Singleplayer,Explo...",83.1,392287.0
2,151603712,Spore,14.9,3,17390.0,SPORE™,"Dec 19, 2008",4.99,"From Single Cell to Galactic God, evolve in a ...",https://shared.akamai.steamstatic.com/store_it...,61124.0,4996.0,1024.0,Maxis™,Electronic Arts,"Single-player,Steam Trading Cards,Family Sharing","Action,Adventure,Casual,RPG,Simulation,Strategy","God Game,Open World,Sandbox,Colony Sim,Explora...",92.4,66120.0


## 3. Build User-Item Rating Matrix

Rows = games, Columns = users (transposed from user-user CF).
We filter to ratings >= 3 for building item similarities.

In [4]:
df_filtered = df[df['rating'] >= MIN_RATING].copy()
print(f'Rows after rating filter (>= {MIN_RATING}): {len(df_filtered):,}')

# user-item matrix (users x games) — kept for prediction lookups
ui_matrix = df_filtered.pivot_table(
    index='user_id', columns='game', values='rating'
)
print(f'User-item matrix: {ui_matrix.shape}  (users x games)')

# item-user matrix (games x users) — used to build item similarities
# Only include popular games (rated by >= MIN_GAME_RATERS users)
game_rater_count = ui_matrix.notna().sum(axis=0)
popular_games    = game_rater_count[game_rater_count >= MIN_GAME_RATERS].index
print(f'Popular games (>= {MIN_GAME_RATERS} raters): {len(popular_games):,} / {ui_matrix.shape[1]:,}')

iu_matrix = ui_matrix[popular_games].T   # (popular_games x users)
print(f'Item-user matrix: {iu_matrix.shape}  (games x users)')

users      = ui_matrix.index.tolist()
all_games  = ui_matrix.columns.tolist()
pop_games  = iu_matrix.index.tolist()     # only popular games

Rows after rating filter (>= 3): 29,450
User-item matrix: (3270, 2167)  (users x games)
Popular games (>= 5 raters): 791 / 2,167
Item-user matrix: (791, 3270)  (games x users)


## 4. Compute Baseline Estimates

Same formula as before, but we use these per-prediction rather than per-user:
```
μ         = global mean
bᵤ        = mean(user u ratings) - μ
bᵢ        = mean(game i ratings) - μ
b(u,i)    = μ + bᵤ + bᵢ
```

In [5]:
mu        = df_filtered['rating'].mean()
user_mean = ui_matrix.mean(axis=1)     # mean rating per user
game_mean = ui_matrix.mean(axis=0)     # mean rating per game (all games, not just popular)
user_bias = (user_mean - mu).to_dict()
game_bias = (game_mean - mu).to_dict()

print(f'Global mean rating (μ): {mu:.4f}')
print(f'User bias  min/max: {min(user_bias.values()):.3f} / {max(user_bias.values()):.3f}')
print(f'Game bias  min/max: {min(game_bias.values()):.3f} / {max(game_bias.values()):.3f}')

Global mean rating (μ): 3.6458
User bias  min/max: -0.646 / 1.354
Game bias  min/max: -0.646 / 1.354


## 5. Compute Item-Item Pearson Similarity Matrix

Each row is a game's rating vector across all users.
We mean-center per item (Pearson), then compute cosine similarity.
Pairs with fewer than `MIN_GAMES_COMMON` shared raters are zeroed out.

In [6]:
# Mean-center each item's ratings across users (item Pearson)
item_mean    = iu_matrix.mean(axis=1)                        # mean per game
iu_centered  = iu_matrix.subtract(item_mean, axis=0)        # center
iu_filled    = iu_centered.fillna(0).values                  # NaN -> 0

# Co-rater count matrix: how many users rated both item i and item j
rated_mask   = iu_matrix.notna().values.astype(np.float32)   # (n_items x n_users)
corater_count = rated_mask @ rated_mask.T                     # (n_items x n_items)

# Pearson similarity
item_sim = cosine_similarity(iu_filled)                       # (n_items x n_items)
item_sim[corater_count < MIN_GAMES_COMMON] = 0
np.fill_diagonal(item_sim, 0)

print(f'Item similarity matrix shape: {item_sim.shape}')
print(f'Non-zero similarities: {(item_sim != 0).sum():,}')
sparsity = 1 - (item_sim != 0).sum() / (item_sim.shape[0] ** 2)
print(f'Sparsity: {sparsity:.2%}  (much denser than user-user was)')

Item similarity matrix shape: (791, 791)
Non-zero similarities: 63,726
Sparsity: 89.81%  (much denser than user-user was)


## 6. Prediction Function

In [7]:
# Index lookups for fast access
pop_game_to_idx = {g: i for i, g in enumerate(pop_games)}


def baseline(user_id, game_name):
    ub = user_bias.get(user_id, 0.0)
    gb = game_bias.get(game_name, 0.0)
    return np.clip(mu + ub + gb, 1, 5)


def predict_item_cf(user_id, candidate_game, user_ratings_series):
    """
    Item-item CF prediction:
      pred(u,i) = b(u,i) + Σ[ sim(i,j) * (r(u,j) - b(u,j)) ] / Σ|sim(i,j)|
    where j iterates over games the user HAS played that are similar to i.
    """
    b_ui = baseline(user_id, candidate_game)

    if candidate_game not in pop_game_to_idx:
        return b_ui

    ci = pop_game_to_idx[candidate_game]
    sims = item_sim[ci]   # similarity of candidate to all other popular games

    # Games user has played that are (a) popular and (b) have nonzero similarity
    played = user_ratings_series.dropna()
    played_popular = [
        (pop_game_to_idx[g], g, r)
        for g, r in played.items()
        if g in pop_game_to_idx and sims[pop_game_to_idx[g]] != 0
    ]

    if not played_popular:
        return b_ui

    # Sort by |sim| descending, take top-K
    played_popular.sort(key=lambda x: abs(sims[x[0]]), reverse=True)
    played_popular = played_popular[:TOP_K_ITEMS]

    numer, denom = 0.0, 0.0
    for ji, g, r_uj in played_popular:
        s   = sims[ji]
        b_j = baseline(user_id, g)
        numer += s * (r_uj - b_j)
        denom += abs(s)

    if denom == 0:
        return b_ui

    return np.clip(b_ui + numer / denom, 1, 5)


def rmse(preds, actuals):
    return np.sqrt(np.mean((np.array(preds) - np.array(actuals)) ** 2))


print('Prediction functions defined.')

Prediction functions defined.


## 7. Evaluate — Train / Test Split

Uses the full 1–5 rating range so RMSE is meaningful across all rating levels.

In [8]:
# Full-range rating matrix for evaluation (not filtered to >= 3)
ui_full = df.pivot_table(
    index='user_id', columns='game', values='rating'
).reindex(index=users, columns=all_games)

known = [
    (ui_matrix.index[ui], ui_matrix.columns[gi], ui_full.iloc[ui, gi])
    for ui in range(len(users))
    for gi in range(len(all_games))
    if not np.isnan(ui_full.iloc[ui, gi])
]
print(f'Total known ratings (full 1–5): {len(known):,}')

train_entries, test_entries = train_test_split(known, test_size=TEST_SIZE, random_state=42)
print(f'Train: {len(train_entries):,}  |  Test: {len(test_entries):,}')

Total known ratings (full 1–5): 57,015
Train: 45,612  |  Test: 11,403


In [9]:
# Build train-only versions of the matrices
train_ui = ui_full.copy()
for user_id, game_name, _ in test_entries:
    train_ui.loc[user_id, game_name] = np.nan

# Filter to >= MIN_RATING for similarity/bias
train_filtered = train_ui.copy()
train_filtered[train_filtered < MIN_RATING] = np.nan

# Train biases
t_mu        = np.nanmean(train_filtered.values)
t_user_mean = train_filtered.mean(axis=1)
t_game_mean = train_filtered.mean(axis=0)
t_user_bias = (t_user_mean - t_mu).fillna(0).to_dict()
t_game_bias = (t_game_mean - t_mu).fillna(0).to_dict()

# Train item-item similarity (popular games only)
t_iu        = train_filtered[popular_games].T
t_item_mean = t_iu.mean(axis=1)
t_iu_cen    = t_iu.subtract(t_item_mean, axis=0).fillna(0).values
t_rated     = t_iu.notna().values.astype(np.float32)
t_corater   = t_rated @ t_rated.T
t_item_sim  = cosine_similarity(t_iu_cen)
t_item_sim[t_corater < MIN_GAMES_COMMON] = 0
np.fill_diagonal(t_item_sim, 0)

# Also build train user-user cosine for comparison
t_ui_cen    = train_filtered.subtract(t_user_mean, axis=0).fillna(0).values
t_user_sim  = cosine_similarity(t_ui_cen)
t_u_rated   = train_filtered.notna().values.astype(np.float32)
t_u_corater = t_u_rated @ t_u_rated.T
t_user_sim[t_u_corater < MIN_GAMES_COMMON] = 0
np.fill_diagonal(t_user_sim, 0)

print('Train matrices ready.')

Train matrices ready.


In [10]:
def t_baseline(user_id, game_name):
    ub = t_user_bias.get(user_id, 0.0)
    gb = t_game_bias.get(game_name, 0.0)
    return np.clip(t_mu + ub + gb, 1, 5)


def t_predict_item_cf(user_id, game_name):
    b_ui = t_baseline(user_id, game_name)
    if game_name not in pop_game_to_idx:
        return b_ui
    ci   = pop_game_to_idx[game_name]
    sims = t_item_sim[ci]

    user_row = train_filtered.loc[user_id] if user_id in train_filtered.index else None
    if user_row is None:
        return b_ui

    played_popular = [
        (pop_game_to_idx[g], g, r)
        for g, r in user_row.dropna().items()
        if g in pop_game_to_idx and sims[pop_game_to_idx[g]] != 0
    ]
    if not played_popular:
        return b_ui

    played_popular.sort(key=lambda x: abs(sims[x[0]]), reverse=True)
    played_popular = played_popular[:TOP_K_ITEMS]

    numer, denom = 0.0, 0.0
    for ji, g, r_uj in played_popular:
        s   = sims[ji]
        b_j = t_baseline(user_id, g)
        numer += s * (r_uj - b_j)
        denom += abs(s)

    if denom == 0:
        return b_ui
    return np.clip(b_ui + numer / denom, 1, 5)


def t_predict_user_cf(user_id, game_name):
    """User-user Pearson CF on train data (for RMSE comparison only)."""
    b_ui = t_baseline(user_id, game_name)
    if user_id not in train_filtered.index or game_name not in train_filtered.columns:
        return b_ui
    ui = list(train_filtered.index).index(user_id)
    gi = list(train_filtered.columns).index(game_name) if game_name in train_filtered.columns else None
    if gi is None:
        return b_ui
    sims         = t_user_sim[ui]
    game_ratings = train_filtered.iloc[:, gi].values
    mask         = ~np.isnan(game_ratings) & (sims != 0)
    if mask.sum() == 0:
        return b_ui
    top_k    = np.argsort(np.abs(sims[mask]))[::-1][:30]
    top_sims = sims[mask][top_k]
    top_rats = game_ratings[mask][top_k]
    top_idx  = np.where(mask)[0][top_k]
    denom    = np.sum(np.abs(top_sims))
    if denom == 0:
        return b_ui
    v_base    = np.array([t_baseline(train_filtered.index[v], game_name) for v in top_idx])
    residuals = top_rats - v_base
    return np.clip(b_ui + np.dot(top_sims, residuals) / denom, 1, 5)


print('Train prediction functions defined.')

Train prediction functions defined.


In [11]:
print('Evaluating on test set... (this may take a few minutes)')

actuals, bl_preds, uu_preds, ii_preds = [], [], [], []

for user_id, game_name, true_r in tqdm(test_entries, desc='Evaluating'):
    actuals.append(true_r)
    bl_preds.append(t_baseline(user_id, game_name))
    uu_preds.append(t_predict_user_cf(user_id, game_name))
    ii_preds.append(t_predict_item_cf(user_id, game_name))

print(f'\n=== Evaluation Results (n={len(test_entries):,} test ratings, full 1–5 scale) ===')
print(f'Baseline estimate only      RMSE: {rmse(bl_preds, actuals):.4f}')
print(f'User-user Pearson CF        RMSE: {rmse(uu_preds, actuals):.4f}  (v2 method for reference)')
print(f'Item-item Pearson CF        RMSE: {rmse(ii_preds, actuals):.4f}  <- our method (v3)')

Evaluating on test set... (this may take a few minutes)


Evaluating:   0%|          | 0/11403 [00:00<?, ?it/s]


=== Evaluation Results (n=11,403 test ratings, full 1–5 scale) ===
Baseline estimate only      RMSE: 1.4553
User-user Pearson CF        RMSE: 1.4672  (v2 method for reference)
Item-item Pearson CF        RMSE: 1.4799  <- our method (v3)


## 8. Generate Recommendations for All Users

In [12]:
print('Generating recommendations...')
popular_games_set = set(pop_games)

raw_scores_per_user = {}

for ui_idx, user_id in enumerate(tqdm(users, desc='Scoring')):
    user_row   = ui_matrix.iloc[ui_idx]            # ratings this user has given
    played     = all_played_by_user.get(user_id, set())
    candidates = [g for g in pop_games if g not in played]

    if not candidates:
        raw_scores_per_user[user_id] = []
        continue

    scores = [
        (g, predict_item_cf(user_id, g, user_row))
        for g in candidates
    ]
    scores.sort(key=lambda x: x[1], reverse=True)
    raw_scores_per_user[user_id] = scores

print('Scoring done.')

Generating recommendations...


Scoring:   0%|          | 0/3270 [00:00<?, ?it/s]

Scoring done.


## 9. Sanity Check — Diversity Before Any Dampening

In [13]:
pre_counts = Counter(
    g for scores in raw_scores_per_user.values()
    for g, _ in scores[:TOP_N]
)
print(f'Unique games in top-{TOP_N} lists: {len(pre_counts)}')
print(f'\nTop 15 most recommended (raw, before any dampening):')
for game, count in pre_counts.most_common(15):
    print(f'  {100*count/len(users):5.1f}%  {game}')

Unique games in top-10 lists: 462

Top 15 most recommended (raw, before any dampening):
   55.5%  Football Manager 2011
   55.4%  Football Manager 2013
   53.1%  Football Manager 2012
   51.9%  Football Manager 2014
   44.6%  Football Manager 2010
   42.8%  Football Manager 2015
   42.0%  Might & Magic Duel of Champions
   29.0%  NBA 2K15
   23.9%  Age of Empires Online
   21.8%  Sid Meier's Civilization IV Beyond the Sword
   18.8%  DARK SOULS II
   17.2%  Call of Duty Modern Warfare 2 - Multiplayer
   15.5%  Dota 2
   14.9%  Arma 3
   14.3%  ARK Survival Evolved


> **Decision point:** If the raw diversity above looks good (no game dominating > ~30% of users), skip the dampening cell below. If Football Manager / X3 are still dominant, run it.

In [14]:
# Optional popularity dampening — run only if raw diversity is still poor
# Set POPULARITY_ALPHA = 0 to disable
POPULARITY_ALPHA = 0.8

max_count = max(pre_counts.values()) if pre_counts else 1

def popularity_penalty(game_name):
    norm = pre_counts.get(game_name, 0) / max_count
    return 1.0 / (1.0 + POPULARITY_ALPHA * norm)

final_scores_per_user = {}
for user_id, scores in raw_scores_per_user.items():
    dampened = [(g, s * popularity_penalty(g)) for g, s in scores]
    dampened.sort(key=lambda x: x[1], reverse=True)
    final_scores_per_user[user_id] = dampened[:TOP_N]

post_counts = Counter(
    g for scores in final_scores_per_user.values()
    for g, _ in scores
)
print(f'Unique games after dampening: {len(post_counts)}')
print(f'\nTop 15 after dampening:')
for game, count in post_counts.most_common(15):
    print(f'  {100*count/len(users):5.1f}%  {game}')

Unique games after dampening: 656

Top 15 after dampening:
   22.2%  Microsoft Flight Simulator X Steam Edition
   22.0%  The Witcher 3 Wild Hunt
   18.8%  Total War SHOGUN 2
   18.8%  Terraria
   16.4%  Sid Meier's Civilization V
   16.1%  Grand Theft Auto V
   14.2%  Might & Magic X - Legacy 
   14.2%  Rust
   13.9%  METAL GEAR SOLID V THE PHANTOM PAIN
   13.5%  Napoleon Total War
   12.3%  Kerbal Space Program
   12.2%  The Elder Scrolls V Skyrim
   12.1%  X3 Albion Prelude
   11.4%  Garry's Mod
   10.9%  Star Trek Online


## 10. Build & Save Output JSON Files

In [25]:
# BEFORE
meta_export_cols = ['app_id', 'cover_image_url', 'genres', 'tags', 'description',
                    'price', 'review_score_pct', 'developers', 'release_date']
games_metadata = {}
for game_name, row in meta_df.iterrows():
    entry = {}
    for col in meta_export_cols:
        if col in row.index:
            val = row[col]
            entry[col] = None if pd.isna(val) else val
    games_metadata[game_name] = entry

# AFTER
meta_export_cols = [
    ('app_id',          'aid'),
    ('cover_image_url', 'img'),
    ('genres',          'g'),
    ('tags',            't'),
    ('description',     'desc'),
    ('price',           'price'),
    ('review_score_pct','rv'),
    ('developers',      'dev'),
    ('release_date',    'rd'),
]
games_metadata = {}
for game_name, row in meta_df.iterrows():
    entry = {}
    for old_col, new_key in meta_export_cols:
        if old_col in row.index:
            val = row[old_col]
            entry[new_key] = None if pd.isna(val) else val
    games_metadata[game_name] = entry

## 10b. Export item_sim for dashboard Pick mode

In [26]:
ITEM_SIM_PATH = Path('output/item_sim.json')

# Store top-20 CF neighbours per game (sparse export, ~400 KB)
item_sim_export = {}
for i, game_name in enumerate(pop_games):
    sims = item_sim[i]
    top_indices = np.argsort(sims)[::-1][:20]
    neighbours = [
        [pop_games[j], round(float(sims[j]), 4)]
        for j in top_indices
        if sims[j] > 0
    ]
    if neighbours:
        item_sim_export[game_name] = neighbours

with open(ITEM_SIM_PATH, 'w', encoding='utf-8') as f:
    json.dump(item_sim_export, f, ensure_ascii=False, separators=(',', ':'))

print(f'Saved {ITEM_SIM_PATH}  ({ITEM_SIM_PATH.stat().st_size/1024:.0f} KB)')
print(f'Games with neighbours: {len(item_sim_export):,} / {len(pop_games):,}')

Saved output\item_sim.json  (321 KB)
Games with neighbours: 705 / 791


## 11. Final Sanity Checks

In [27]:
import statistics

lengths    = [len(v) for v in recommendations.values()]
all_scores = [r['pr'] for v in recommendations.values() for r in v]
all_flat   = [r['n']  for v in recommendations.values() for r in v]
exact5     = sum(1 for s in all_scores if s == 5.0)

fingerprints = Counter(tuple(r['n'] for r in v) for v in recommendations.values())

print(f'Users with full {TOP_N} recommendations: {lengths.count(TOP_N):,} / {len(recommendations):,}')
print(f'Unique games recommended:               {len(set(all_flat))}')
print(f'Distinct top-{TOP_N} lists:             {len(fingerprints)} / {len(recommendations)}')
print(f'\nScore stats:')
print(f'  min={min(all_scores):.4f}  max={max(all_scores):.4f}  mean={statistics.mean(all_scores):.4f}  stdev={statistics.stdev(all_scores):.4f}')
print(f'  Exactly 5.0: {exact5}/{len(all_scores)} ({100*exact5/len(all_scores):.1f}%)')

covered = sum(1 for v in recommendations.values() for r in v if r.get('img'))
total   = sum(len(v) for v in recommendations.values())
print(f'\nRec slots with cover image: {covered}/{total} ({100*covered/total:.1f}%)')

print(f'\nTop #1 recommendation diversity:')
top1 = Counter(v[0]['n'] for v in recommendations.values() if v)
for game, count in top1.most_common(10):
    print(f'  {100*count/len(recommendations):5.1f}%  {game}')

Users with full 10 recommendations: 3,270 / 3,270
Unique games recommended:               656
Distinct top-10 lists:             2845 / 3270

Score stats:
  min=3.2692  max=5.0000  mean=4.4538  stdev=0.4756
  Exactly 5.0: 2274/32700 (7.0%)

Rec slots with cover image: 22631/32700 (69.2%)

Top #1 recommendation diversity:
    8.1%  Microsoft Flight Simulator X Steam Edition
    6.5%  The Witcher 3 Wild Hunt
    5.4%  Might & Magic X - Legacy 
    4.9%  Terraria
    2.4%  Total War SHOGUN 2
    2.1%  The Elder Scrolls V Skyrim
    2.0%  Napoleon Total War
    1.8%  Sid Meier's Civilization V
    1.6%  X3 Albion Prelude
    1.3%  Grand Theft Auto V


In [28]:
# Show recommendations for a specific user — change SHOW_USER to any user_id
SHOW_USER = str(users[0])

print(f'Top {TOP_N} recommendations for user {SHOW_USER}:')
print(f'{"#":<4} {"Game":42s} {"Score":>8} {"Genres":28s} {"Review%":>8}')
print('-' * 96)
for i, r in enumerate(recommendations[SHOW_USER], 1):
    genres  = str(r.get('g') or '')[:26]
    rev_pct = r.get('rv')
    rev_str = f"{rev_pct:.1f}%" if rev_pct else 'N/A'
    print(f"{i:<4} {r['n']:42s} {r['pr']:>8.4f} {genres:28s} {rev_str:>8}")

print(f'\nThis user\'s play history (games rated >= {MIN_RATING}):')
played = ui_matrix.loc[int(SHOW_USER)].dropna().sort_values(ascending=False)
for game, rating in played.items():
    print(f'  [{int(rating)}] {game}')

Top 10 recommendations for user 5250:
#    Game                                          Score Genres                        Review%
------------------------------------------------------------------------------------------------
1    H1Z1                                         4.9912                                   N/A
2    Wargame European Escalation                  4.9650 Indie,Simulation,Strategy       78.0%
3    Wargame Red Dragon                           4.9650 Indie,Simulation,Strategy       88.8%
4    Men of War Assault Squad 2                   4.9628 Action,Simulation,Strategy      91.3%
5    Prison Architect                             4.9557 Indie,Simulation,Strategy       89.1%
6    Train Fever                                  4.9455 Casual,Indie,Simulation         64.3%
7    Kerbal Space Program                         4.9112 Indie,Simulation                95.2%
8    Star Trek Online                             4.9069 Massively Multiplayer,RPG,      79.1%
9    Supre

## 12. Export play_history.json for dashboard

In [29]:
HISTORY_PATH = Path('output/play_history.json')

# Build per-user play history from the full dataframe
# Include: game name, hours_played, rating, and metadata if available
play_history = {}

for user_id, group in df.groupby('user_id'):
    games_played = []
    for _, row in group.sort_values('hours_played', ascending=False).iterrows():
        entry = {
            'game':         row['game'],
            'hours':        round(float(row['hours_played']), 1),
            'rating':       int(row['rating']),
        }
        # Attach metadata if available
        if row['game'] in meta_df.index:
            meta_row = meta_df.loc[row['game']]
            entry['img'] = None if pd.isna(meta_row.get('cover_image_url')) else meta_row.get('cover_image_url')
            entry['genres'] = None if pd.isna(meta_row.get('genres')) else meta_row.get('genres')
            entry['app_id'] = None if pd.isna(meta_row.get('app_id')) else int(meta_row.get('app_id'))
        else:
            entry['img']    = None
            entry['genres'] = None
            entry['app_id'] = None
        games_played.append(entry)
    play_history[str(user_id)] = games_played

with open(HISTORY_PATH, 'w', encoding='utf-8') as f:
    json.dump(play_history, f, ensure_ascii=False, separators=(',', ':'))

print(f'Saved {HISTORY_PATH}  ({HISTORY_PATH.stat().st_size/1024/1024:.2f} MB)')
print(f'Users exported: {len(play_history):,}')
print(f'Sample ({list(play_history.keys())[0]}): {play_history[list(play_history.keys())[0]][:2]}')

Saved output\play_history.json  (10.34 MB)
Users exported: 3,466
Sample (5250): [{'game': 'Cities Skylines', 'hours': 144.0, 'rating': 5, 'img': 'https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/255710/header.jpg?t=1764774646', 'genres': 'Simulation,Strategy', 'app_id': 255710}, {'game': 'Deus Ex Human Revolution', 'hours': 62.0, 'rating': 4, 'img': None, 'genres': None, 'app_id': None}]
